In [1]:
import argparse
import os
import pickle
import pprint
import csv

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "2.2.4":
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:

from kawada_env_cnoid import KawadaBaseEnvChoreonoid as RL_Env

In [3]:

# from bex24_env_cnoid import RLEnvChoreonoid as RL_Env

In [4]:
# exp_name = 'kawada-walking-1001'
# ckpt = 1000

In [5]:
# 任意設定項目
exp_name = 'ishiki-walking-no-vel'
ckpt = 2000

action_scale = 0.0 # 動作のスケールを調整
# 関節別スケーリング
joint_scales = torch.tensor([
    0.5, 0.5, 0.8,  # 右脚腰関節：50%, 50%, 80%
    0.7, 0.9, 0.5,  # 右脚膝・足首：70%, 90%, 50%
    0.5, 0.5, 0.8,  # 左脚腰関節：50%, 50%, 80%
    0.7, 0.9, 0.5   # 左脚膝・足首：70%, 90%, 50%
])

In [6]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}
env_cfg["rotorInertia"] = 0.1
env_cfg["kp"] = 1000 # 2000
env_cfg["kd"] = 100 # 500

In [7]:
env = RL_Env(
        num_envs=1,
        env_cfg=env_cfg,
        obs_cfg=obs_cfg,
        reward_cfg=reward_cfg,
        command_cfg=command_cfg,
        # dt=env_cfg['dt'],
        dt=0.005, # 0.01
        # substeps=env_cfg['substeps'],
        substeps=5,
        show_viewer=True,
    )

In [8]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

Actor MLP: Sequential(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: Sequential(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)


/usr/local/lib/python3.10/dist-packages/_distutils_hack/__init__.py:53: UserWarning: Reliance on distutils from stdlib is deprecated. Users must rely on setuptools to provide the distutils module. Avoid importing distutils or import setuptools first, and avoid setting SETUPTOOLS_USE_DISTUTILS=stdlib. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


In [9]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []

# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv'
os.makedirs('obs_data', exist_ok=True)

In [10]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print(obs)
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(obs.cpu().numpy().flatten())
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")

cnt : 0
Original actions :  tensor([[ 1.2061,  1.8438, -0.4992,  2.1252, -1.8702,  1.0577,  0.2597,  1.6540,
         -0.4176, -0.0076, -1.8781,  1.0838]], device='cuda:0')
Scaled actions :  tensor([[0., 0., -0., 0., -0., 0., 0., 0., -0., -0., -0., 0.]],
       device='cuda:0')


/userdir/samples/../irsl_rl/rl_env_base.py:96: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/samples/../irsl_rl/rl_env_cnoid.py:85: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  self.dof_pos = torch.tensor([sbody.angleVector()]).to(torch.float32).to(self.device)


tensor([[-7.1424e-19,  4.4928e-09, -2.6620e-19,  8.8725e-11,  6.8571e-21,
         -1.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00, -6.9908e-21,
          1.7662e-20,  0.0000e+00,  0.0000e+00,  0.0000e+00, -2.9618e-20,
         -7.5170e-21,  1.7230e-20,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         -9.0414e-22,  3.4836e-20,  3.7749e-19, -5.9017e-09,  9.1561e-09,
         -5.7548e-09, -2.3299e-19, -7.1457e-20,  3.3478e-19, -5.9017e-09,
          9.1561e-09, -5.7548e-09,  2.7905e-20,  0.0000e+00,  0.0000e+00,
         -0.0000e+00,  0.0000e+00, -0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00, -0.0000e+00, -0.0000e+00, -0.0000e+00,  0.0000e+00]],
       device='cuda:0')
torques: [ 2.55362686e-17 -6.74320349e-16 -4.57658356e-07  6.69469080e-06
 -1.42313827e-07  4.97024096e-16  2.46396784e-16 -5.97857560e-16
 -4.57658356e-07  6.69469080e-06 -1.42313827e-07  1.26804872e-17]
データ収集: step 1


In [11]:
# 既存のforループを置き換え
num_steps = 4000
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        
        # アクションスケーリング
        scaled_actions = actions * action_scale
        
        torques = env.sim.sbody.getTorques()
        obs, rews, dones, infos = env.step(scaled_actions)  # スケール済みを使用
        
        # データを記録
        step_data.append(cnt)
        obs_data.append(obs.cpu().numpy().flatten())
        torque_data.append(torques.copy())
        
        # デバッグ表示（最初の数ステップのみ）
        if i < 3:
            print(f"Step {i}: Original action max={actions.max():.3f}, "
                  f"Scaled action max={scaled_actions.max():.3f}")
        
        if i % 20 == 0:
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected with action_scale={action_scale}")

Step 0: Original action max=2.343, Scaled action max=0.000
Step 1/4000, Total steps: 1
Step 1: Original action max=2.286, Scaled action max=0.000
Step 2: Original action max=2.249, Scaled action max=0.000
Step 21/4000, Total steps: 21
Step 41/4000, Total steps: 41
Step 61/4000, Total steps: 61
Step 81/4000, Total steps: 81
Step 101/4000, Total steps: 101
Step 121/4000, Total steps: 121
Step 141/4000, Total steps: 141
Step 161/4000, Total steps: 161
Step 181/4000, Total steps: 181
Step 201/4000, Total steps: 201
Step 221/4000, Total steps: 221
Step 241/4000, Total steps: 241
Step 261/4000, Total steps: 261
Step 281/4000, Total steps: 281
Step 301/4000, Total steps: 301
Step 321/4000, Total steps: 321
Step 341/4000, Total steps: 341
Step 361/4000, Total steps: 361
Step 381/4000, Total steps: 381
Step 401/4000, Total steps: 401
Step 421/4000, Total steps: 421
Step 441/4000, Total steps: 441
Step 461/4000, Total steps: 461
Step 481/4000, Total steps: 481
Step 501/4000, Total steps: 501
Ste

In [12]:
# for i in range(500):
#     obs, _ = env.reset()
#     with torch.no_grad():
#         actions = policy(obs)
#         obs, rews, dones, infos = env.step(actions)

In [13]:
env.sim.stop()

In [14]:

# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data/cnoid_{exp_name}_ckpt{ckpt}_scale{action_scale}.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

シンプル版を保存: obs_data/cnoid_ishiki-walking-no-vel_ckpt2000_scale0.25.csv
データ形状: (21, 58)
